<a href="https://colab.research.google.com/github/josenomberto/UTEC-CDIAV3-IAFUND/blob/main/laboratorio_3_alum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCD8009: Data Discovery - Laboratorio 3

**Integrantes**

| N° | Código | Nombres |  Contribución (0% - 100%) |
|----|--------|---------|---------------------------|
| 1  |        | José Carlos Nomberto        | 100                           |
| 2  |        |         |                           |
| 3  |        |         |                           |
| 4  |        |         |                           |

### Indicaciones

- El laboratorio podrá resolverse de manera **individual o en equipos de hasta cuatro (4) personas**. Deberán completar los datos de todos los integrantes, y **una sola persona realizará la entrega del archivo ipynb**.

- Salvo que se indique explícitamente lo contrario, no se prohibe el uso de herramientas de Inteligencia Artificial Generativa, siempre que los integrantes comprendan y puedan explicar el proceso y los resultados obtenidos. **Las respuestas no deben consistir en transcripciones literales de resultados generados por estas herramientas, sino evidenciar comprensión del tema por parte del estudiante o del equipo.**

- En caso de utilizar herramientas de IA Generativa, cada equipo es responsable de verificar la coherencia de las respuestas presentadas. Si se detectan errores, inconsistencias o falta de comprensión, la pregunta podrá ser anulada sin derecho a reclamo.

- En todos los casos, deberá completarse la **Declaración de Uso de IA Generativa.**

- Pueden agregar libremente celdas de código o de Markdown según lo consideren conveniente.

### Declaración de uso de IA Generativa
- Indicar de manera breve la(s) herramienta(s) y/o modelo(s) de IA Generativa utilizados, especificando en qué pregunta(s) se emplearon y con qué propósito.
- En caso no se haya usado, también indicarlo.

## INICIO DEL LABORATORIO

### Parte 1: Transformaciones (4 puntos)

En clase discutimos la posibilidad de aplicar transformaciones a las variables. Estas transformaciones pueden ayudar a normalizar la distribución y facilitar la detección de valores atípicos, especialmente en datos con colas largas o asimetría. Algunas de las [transformaciones más comunes](https://www.marsja.se/transform-skewed-data-using-square-root-log-box-cox-methods-in-python/) incluyen:

- [Transformación logarítmica](https://medium.com/@kyawsawhtoon/log-transformation-purpose-and-interpretation-9444b4b049c9)
- Transformación de raíz cuadrada
- [Transformación de Box-Cox](https://es.wikipedia.org/wiki/Transformaci%C3%B3n_Box-Cox)
- [Transformación de Yeo-Johnson](https://feature-engine.trainindata.com/en/1.8.x/user_guide/transformation/YeoJohnsonTransformer.html)

| Transformación    | Reduce colas largas | Permite valores negativos | Mejora la normalidad |
|------------------|--------------------|--------------------------|---------------------|
| **Logarítmica**  | ✅ Sí              | ❌ No                    | ✅ Sí               |
| **Raíz cuadrada** | ✅ Moderado       | ❌ No                    | ✅ Parcialmente     |
| **Box-Cox**      | ✅ Automático      | ❌ No                    | ✅ Sí               |
| **Yeo-Johnson**  | ✅ Automático      | ✅ Sí                    | ✅ Sí               |

Una manera práctica (rule-of-thumb) es usar el siguiente criterio:

- Si los datos son sesgados a la derecha: Logarítmica, Box-Cox o raíz cuadrada.
- Si hay valores negativos: Yeo-Johnson o raíz cúbica.
- Si no se sabe qué transformación aplicar: Box-Cox o Yeo-Johnson.

Vamos a continuar con el dataset `Salaries.csv` del laboratorio 2. Nuevamente nuestra variable de interés será "TotalPayBenefits" y vamos a volver considerar los salarios de los empleados en el año 2014 y Status "Full Time" (FT) únicamente. Ejecute estos filtros para el análisis. Agregue 4 columnas al dataset, cada una con su respectiva transformación:

- Logarítmica
- Raíz cuadrada
- Box-Cox
- Yeo-Johnson

Grafique la distribución original + las 4 distribuciones resultantes y comente qué observa. ¿Se ven diferencias entre los 4 métodos? (mire tanto la forma de la distribución resultante como la escala)



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from pathlib import Path
from IPython.display import Image

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.neighbors import LocalOutlierFactor
from scipy.spatial.distance import mahalanobis
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from scipy.stats import chi2
from sklearn.cluster import DBSCAN
from sklearn.datasets import fetch_openml

In [ ]:
salaries = pd.read_csv("Salaries.csv", low_memory=False)
salaries.head()

,Id,EmployeeName,JobTitle,BasePay,OvertimePay,OtherPay,Benefits,TotalPay,TotalPayBenefits,Year,Notes,Agency,Status
0,1,NATHANIEL FORD,GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY,167411.18,0.0,400184.25,NaN,567595.43,567595.43,2011,NaN,San Francisco,NaN
1,2,GARY JIMENEZ,CAPTAIN III (POLICE DEPARTMENT),155966.02,245131.88,137811.38,NaN,538909.28,538909.28,2011,NaN,San Francisco,NaN
2,3,ALBERT PARDINI,CAPTAIN III (POLICE DEPARTMENT),212739.13,106088.18,16452.6,NaN,335279.91,335279.91,2011,NaN,San Francisco,NaN
3,4,CHRISTOPHER CHONG,WIRE ROPE CABLE MAINTENANCE MECHANIC,77916.0,56120.71,198306.9,NaN,332343.61,332343.61,2011,NaN,San Francisco,NaN
4,5,PATRICK GARDNER,"DEPUTY CHIEF OF DEPARTMENT,(FIRE DEPARTMENT)",134401.6,9737.0,182234.59,NaN,326373.19,326373.19,2011,NaN,San Francisco,NaN


### Parte 2: Detección de datos atípicos multivariados (10 puntos)

#### 2.1 Parte teórica (5 puntos)

Responda las siguientes preguntas:

a) En el contexto de la maldición de la dimensionalidad, ¿cómo cambia la definición o interpretación de un atípico en espacios de alta dimensión y por qué ciertos métodos de detección de datos atípicos pueden volverse ineficaces? ¿Qué estrategias o enfoques pueden emplearse para mitigar este problema? (2 puntos)

b) En clase revisamos distintos métodos de detección de atípicos multivariantes, pero dicha lista no es extensiva. Otro método conocido, el cual se basada en densidades, es [**LOF (Local Outlier Factor)**](https://en.wikipedia.org/wiki/Local_outlier_factor), que estima el grado de atipicidad de un punto comparando su densidad local con la densidad de sus vecinos más cercanos. A diferencia de enfoques globales, LOF permite identificar outliers locales, es decir, puntos que son anómalos respecto a su vecindario inmediato, incluso si no lo son a nivel global. Se adjunta como hipervínculo su [paper](https://dl.acm.org/doi/epdf/10.1145/335191.335388) y su [implementación en scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.LocalOutlierFactor.html). Vamos a incluir la aplicación de este método en la pregunta 2.2.

Asimismo, existen métodos inspirados en [DBSCAN](https://file.biolab.si/papers/1996-DBSCAN-KDD.pdf) que buscan atacar sus debilidades, estos son:

- **HDBSCAN**, siglas de **H**ierarchical **D**ensity-**B**ased **S**patial **C**lustering of **A**pplications with **N**oise. Se adjunta como hipervínculo su [paper](https://arxiv.org/pdf/1911.02282).
- **OPTICS**, siglas de **O**rdering **P**oints **T**o **I**dentify the **C**lustering **S**tructure. Se adjunta como hipervínculo su [paper](https://dl.acm.org/doi/epdf/10.1145/304181.304187).

Lea (ya sea a través de los papers o en otras fuentes en internet) y, en sus propias palabras, explique qué mejoran y cómo lo hacen con respecto al DBSCAN original. (3 puntos)

a)


b)

#### 2.2 Parte práctica (5 puntos)

Vamos a emplear un dataset del sector inmobiliario, en particular, del dataset House Prices de Ames, Iowa, del cual se toman dos variables:

- LotArea: Área total del lote en pies cuadrados.
- GrLivArea: Área habitable sobre el suelo en pies cuadrados.

El objetivo es detectar valores atípicos multivariados en estos datos utilizando los siguientes métodos:

1. **Distancia de Mahalanobis** - α = 0.025
2. **Isolation Forest** - IsolationForest(contamination=0.05, random_state=42)
3. **Local Outlier Factor (LOF)** - LocalOutlierFactor(n_neighbors=20)
4. **DBSCAN** - DBSCAN(eps=350, min_samples=5)

Emplee los argumentos brindados. Realice 4 gráficos de dispersión de las variables LotArea vs GrLivArea (uno por cada método), donde los puntos rojos representen los valores atípicos detectados por cada método y los puntos azules los valores normales. Analice y comente sobre los resultados. ¿Los datos atípicos detectados por estos métodos coinciden con lo que hubieran determinado intuitivamente de manera visual?

In [ ]:
data = fetch_openml(name="house_prices", as_frame=True)["data"]
data = data[["LotArea", "GrLivArea"]].dropna().astype(float)
data.head()

,LotArea,GrLivArea
0,8450.0,1710.0
1,9600.0,1262.0
2,11250.0,1786.0
3,9550.0,1717.0
4,14260.0,2198.0


### Parte 3: Calidad de datos (6 puntos)

Se presenta un dataset que contiene 200 solicitudes de crédito personal. Asimismo, se proporciona un diccionario de datos, en el cual se especifica para cada variable: el nombre de la columna, el tipo de dato, su descripción, si admite valores nulos y si posee alguna restricción.

El objetivo de este ejercicio es implementar las 10 reglas de calidad de datos indicadas en la tabla. Cabe señalar que, en base al diccionario de datos, podrían definirse muchas más reglas; sin embargo, para efectos de esta actividad, solo se deben considerar las 10 especificadas.

Como resultado final, se espera una tabla o un reporte (prints) que incluya:

- La cantidad y % de registros que incumplen cada regla de calidad.
- El id_solicitud y los valores correspondientes a los registros que presentan incumplimientos.

La implementación puede realizarse utilizando `Great Expectations` o mediante validaciones ad-hoc en código.

#### Diccionario de datos

| # | Columna | Tipo | Descripción | Ejemplo | Nulos permitidos | Restricción / Dominio |
|---|---------|------|-------------|---------|------------------|-----------------------|
| 1 | `id_solicitud` | str | Identificador único de la solicitud | `SOL-00001` | No | Formato `SOL-NNNNN` |
| 2 | `dni` | str | DNI del solicitante (Perú) | `12345678` | No | Exactamente 8 dígitos numéricos |
| 3 | `nombre` | str | Nombre completo del solicitante | `Ana García López` | No | Texto libre, no vacío |
| 4 | `fecha_nacimiento` | date | Fecha de nacimiento | `1990-03-15` | No | Fecha válida; coherente con `edad` |
| 5 | `edad` | int | Edad declarada (años) | `34` | No | Entre 18 y 80 inclusive |
| 6 | `email` | str | Correo electrónico de contacto | `ana@gmail.com` | No | Formato RFC estándar |
| 7 | `telefono` | str | Teléfono celular (Perú) | `987654321` | No | 9 dígitos, inicia en `9` |
| 8 | `ingreso_mensual` | float | Ingreso neto mensual declarado (S/) | `3500.00` | No | > 0 |
| 9 | `monto_solicitado` | float | Monto del crédito solicitado (S/) | `15000.00` | No | > 0 |
| 10 | `plazo_meses` | int | Plazo del crédito en meses | `36` | No | Uno de: {12, 24, 36, 48, 60} |
| 11 | `cuota_mensual` | float | Cuota mensual del crédito (S/) | `512.50` | No | > 0; ≤ 30 % del `ingreso_mensual` |
| 12 | `tipo_empleo` | str | Situación laboral del solicitante | `dependiente` | No | {`dependiente`, `independiente`, `desempleado`} |
| 13 | `empresa` | str | Nombre del empleador | `Empresa_SAC_012` | Sí (si no es dependiente) | Obligatorio cuando `tipo_empleo = dependiente` |
| 14 | `estado_solicitud` | str | Estado actual del trámite | `aprobada` | No | {`aprobada`, `rechazada`, `en_revision`} |
| 15 | `fecha_solicitud` | date | Fecha en que se registró la solicitud | `2024-03-20` | No | Fecha válida |
| 16 | `fecha_resolucion` | date | Fecha en que se resolvió la solicitud | `2024-03-28` | Sí (si aún en revisión) | Cuando existe: ≥ `fecha_solicitud` |

#### Reglas de calidad a implementar

| ID | Columna(s) | Descripción |
|----|-----------|-------------|
| **R1** | `dni` | El DNI peruano tiene exactamente 8 dígitos numéricos |
| **R2** | `email` | Estructura `usuario@dominio.ext` |
| **R3** | `telefono` |  Celular peruano: 9 dígitos, inicia en `9` |
| **R4** | `edad` | Menores de 18 no pueden contratar; política de riesgo limita a 80 |
| **R5** | `ingreso_mensual` | Un ingreso ≤ 0 es imposible para calificar al crédito |
| **R6** | `tipo_empleo` | Solo se aceptan tres valores codificados en el sistema |
| **R7** | `tipo_empleo` + `empresa`  | Si es `dependiente`, debe declarar obligatoriamente su empleador |
| **R8** | `fecha_solicitud` + `fecha_resolucion`  | Una solicitud no puede resolverse antes de haberse registrado |
| **R9** | `edad` + `fecha_nacimiento`  | La edad declarada debe ser coherente con la fecha de nacimiento (±1 año) |
| **R10** | `cuota_mensual` + `ingreso_mensual`  | La cuota no puede superar el 30 % del ingreso neto (Reglamento SBS Perú) |

In [ ]:
df_creditos=pd.read_csv("solicitudes_credito.csv", parse_dates=["fecha_nacimiento", "fecha_solicitud", "fecha_resolucion"])
df_creditos.head()

,id_solicitud,dni,nombre,fecha_nacimiento,edad,email,telefono,ingreso_mensual,monto_solicitado,plazo_meses,cuota_mensual,tipo_empleo,empresa,estado_solicitud,fecha_solicitud,fecha_resolucion
0,SOL-00001,95822412,Sofía Rivera Pérez,1965-01-01,60,usuario0@yahoo.com,953389073,3306.20,7047.62,48,207.02,dependiente,Empresa_EIRL_000,aprobada,2024-08-25,2024-08-28
1,SOL-00002,24942603,Jorge Torres Ramírez,1975-01-02,50,usuario1@gmail.com,925558733,2670.08,18637.76,36,673.80,dependiente,Empresa_SA_001,aprobada,2024-10-15,2024-10-23
2,SOL-00003,13356886,Luis Martínez Martínez,1989-01-01,36,usuario2@outlook.com,964247457,5051.93,22101.36,24,1103.39,dependiente,Empresa_EIRL_002,en_revision,2024-10-23,2024-11-21
3,SOL-00004,46913810,Luis Pérez García,1961-01-01,64,usuario3@outlook.com,979067939,2755.68,5645.25,60,143.35,dependiente,Empresa_SAC_003,aprobada,2024-08-17,2024-08-18
4,SOL-00005,42868828,Miguel Torres Martínez,1996-01-02,29,usuario4@gmail.com,910154493,4273.86,30730.78,60,780.36,dependiente,Empresa_SA_004,en_revision,2024-12-14,2024-12-27
